# Limpieza de answers — dataset_100_curado
Usa un modelo local via Ollama para extraer solo el bloque de código Cisco válido de cada respuesta.

**Requisitos:**
```bash
pip install ollama pandas tqdm
ollama pull qwen2.5:14b   # o el modelo que tengas
```

In [2]:
import re
import pandas as pd
from tqdm.notebook import tqdm

try:
    import ollama
except ImportError:
    raise SystemExit('Falta el SDK: pip install ollama')

## Configuración

In [3]:
MODEL       = 'qwen2.5:14b'          # cambia por el modelo que tengas en Ollama
INPUT_PATH  = 'dataset_100_curado.csv'
OUTPUT_PATH = 'dataset_100_cleaned.csv'

## Prompt

In [4]:
SYSTEM_PROMPT = """You are a Cisco IOS configuration expert and dataset cleaner.
Your task is to extract the single most complete and general Cisco command block from a network configuration answer.

Rules:
- Return ONLY the Cisco IOS commands, nothing else.
- If the answer contains multiple code blocks (e.g. a general template and a specific example), keep ONLY the most general and complete one. Do NOT merge them.
- If a command appears repeated (once as explanation, once as example), keep it only once.
- Do NOT invent, add, modify, or complete any command. Copy them exactly as they appear in the original.
- Remove all prose, explanations, notes, and markdown formatting (no ```, no Router(config)# labels if they are not in the original).
- If there is no valid Cisco command block, return exactly: NO_CODE

Output format — raw commands only, one per line:
command1
command2
..."""

## Verificar Ollama

In [9]:
try:
    modelos = ollama.list()
    print('Ollama OK. Modelos disponibles:')
    for m in modelos['models']:
        print(' -', m['name'])
except Exception:
    raise SystemExit('Ollama no está corriendo. Ejecuta: ollama serve')

Ollama OK. Modelos disponibles:


SystemExit: Ollama no está corriendo. Ejecuta: ollama serve

## Cargar dataset

In [10]:
df = pd.read_csv(INPUT_PATH)
print(f'Dataset cargado: {len(df)} filas')
print(df['category'].value_counts())
df.head(2)

Dataset cargado: 100 filas
category
SECURITY        20
ROUTING         20
MONITORING      20
QOS             20
CONNECTIVITY    20
Name: count, dtype: int64


,id,question,answer,category
0,2301,What is the configuration to automatically arc...,```cisco\r\ncrypto pki auto-rollover 00 00 24\...,SECURITY
1,2728,How do you configure the number of simultaneou...,To configure the number of simultaneous packet...,SECURITY


## Función de limpieza

In [11]:
def extract_clean_answer(model: str, answer: str) -> str:
    """Llama al modelo y devuelve el bloque de comandos limpio."""
    try:
        response = ollama.chat(
            model=model,
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': answer},
            ],
            options={
                'temperature': 0.0,
                'num_predict': 512,
            },
        )
        raw = response['message']['content'].strip()
        # Quitar backticks si el modelo los incluyó de todos modos
        raw = re.sub(r'```[a-zA-Z]*\n?', '', raw).strip('`').strip()
        return raw if raw else 'NO_CODE'

    except Exception as e:
        print(f'  [ERROR] {e}')
        return 'ERROR'

## Pipeline de limpieza

In [12]:
cleaned = []

for _, row in tqdm(df.iterrows(), total=len(df), desc='Limpiando answers'):
    clean = extract_clean_answer(MODEL, str(row['answer']))
    cleaned.append(clean)

df['answer_clean'] = cleaned
print('Listo.')

Limpiando answers:   0%|          | 0/100 [00:00<?, ?it/s]

Listo.


## Guardar resultado

In [13]:
no_code = (df['answer_clean'] == 'NO_CODE').sum()
error   = (df['answer_clean'] == 'ERROR').sum()
ok      = len(df) - no_code - error
print(f'OK       : {ok}')
print(f'NO_CODE  : {no_code}')
print(f'ERROR    : {error}')

df.to_csv(OUTPUT_PATH, index=False)
print(f'\nGuardado en: {OUTPUT_PATH}')
df.head(3)

OK       : 85
NO_CODE  : 15
ERROR    : 0

Guardado en: dataset_100_cleaned.csv


,id,question,answer,category,answer_clean
0,2301,What is the configuration to automatically arc...,```cisco\r\ncrypto pki auto-rollover 00 00 24\...,SECURITY,crypto pki auto-rollover 00 00 24
1,2728,How do you configure the number of simultaneou...,To configure the number of simultaneous packet...,SECURITY,parameter-map type inspect global max-in-progr...
2,3004,What is the first step in configuring an IPsec...,To configure an IPsec profile for DMVPN on Cis...,SECURITY,crypto ipsec transform-set MY_TRANSFORM esp-3d...


## Inspección manual (opcional)

In [ ]:
# Compara original vs limpio para una fila
idx = 0
row = df.iloc[idx]
print('=== PREGUNTA ===')
print(row['question'])
print('\n=== ANSWER ORIGINAL ===')
print(row['answer'])
print('\n=== ANSWER LIMPIO ===')
print(row['answer_clean'])